# Pipeline Step Debugging

This notebook demonstrates how to use the AutoCut-Agent notebook SDK to:

1. Build a pipeline step by step
2. Inspect each step's configuration
3. Test conditions before running
4. Edit step commands interactively
5. Save the final pipeline as a template

**All changes persist to the database** — the GUI, CLI, and API will see them.

In [ ]:
# Install nest_asyncio if not present (needed for Jupyter)
# !pip install nest_asyncio

from agent.notebook import Session

# Connect to the database (default: agent.db in current directory)
s = Session()
# Or use a specific database:
# s = Session("sqlite+aiosqlite:///path/to/agent.db")
print(s)

## 1. Build a Pipeline Manually

Define steps as dicts, each with `name`, `command_type`, and `command_template`.
Dependencies are declared via `depends_on_names`.

In [ ]:
# Define a video editing pipeline step by step
pipeline, steps = s.compiler.from_steps(
    [
        {
            "name": "resize",
            "command_type": "ffmpeg",
            "command_template": "ffmpeg -i {video_path} -vf scale=1280:720 -c:a copy {output_dir}/resized.mp4",
            "condition": "resolution != 720",  # skip if already 720p
        },
        {
            "name": "scene_detect",
            "command_type": "python",
            "command_template": "python scene_detect.py --input {output_dir}/resized.mp4 --threshold 0.3",
            "depends_on_names": ["resize"],
        },
        {
            "name": "face_detection",
            "command_type": "python",
            "command_template": "python facedetection.py --input {clip_path} --min-faces 1",
            "depends_on_names": ["scene_detect"],
            "fan_out_on": "clips",  # run on each clip in parallel
        },
        {
            "name": "assemble",
            "command_type": "ffmpeg",
            "command_template": "ffmpeg -f concat -i {filtered_list} -c copy {output_dir}/final.mp4",
            "depends_on_names": ["face_detection"],  # fan-in: waits for all clips
        },
    ],
    name="Video Editing Workflow",
    inputs={"video_path": "/data/input.mp4", "output_dir": "/data/output"},
)

print(f"Created pipeline with {len(steps)} steps")

## 2. Inspect the Pipeline

View the full pipeline structure, including dependencies and conditions.

In [ ]:
# Overview of the entire pipeline
s.pipelines.inspect(pipeline, steps)

In [ ]:
# Detailed view of a single step
s.pipelines.inspect_step(
    steps[0],
    context={"resolution": 1080}  # test the condition against real data
)

## 3. Test Conditions Before Running

Validate that your step conditions behave correctly with real context data
**before** deploying the pipeline.

In [ ]:
# The resize step should EXECUTE when resolution is not 720
s.conditions.explain("resolution != 720", {"resolution": 1080})

In [ ]:
# The resize step should SKIP when resolution IS 720
s.conditions.explain("resolution != 720", {"resolution": 720})

In [ ]:
# Test multiple conditions at once against a realistic context
context = {
    "resolution": 1080,
    "duration": 120,
    "has_audio": True,
    "fps": 30,
    "codec": "h264",
}

conditions = [
    "resolution != 720",
    "duration > 60",
    "has_audio",
    "fps >= 24 and fps <= 60",
    "duration > 300",   # this one should be False
]

for expr, result in s.conditions.test_batch(conditions, context):
    status = "EXECUTE" if result else "SKIP"
    print(f"  {status:7s}  {expr}")

## 4. Edit Steps Interactively

Fix command templates or conditions without rebuilding the entire pipeline.

In [ ]:
# Tweak the face detection step command
print("BEFORE:")
print(f"  {steps[2].command_template}")

s.pipelines.edit_step(
    steps[2],
    command_template="python facedetection.py --input {clip_path} --min-faces 2 --confidence 0.7",
    timeout=1200,  # give it more time
)

print("\nAFTER:")
print(f"  {steps[2].command_template}")
print(f"  timeout: {steps[2].timeout}s")

In [ ]:
# Add a condition to the scene_detect step
s.pipelines.edit_step(steps[1], condition="duration > 10")

# Verify it works
s.conditions.explain(steps[1].condition, {"duration": 5})

## 5. Save as Template (Persists to DB)

Save the debugged pipeline as a reusable template. The GUI, API, and CLI
can clone it with different inputs later.

In [ ]:
# Save to database
template = s.pipelines.save_template(
    pipeline=pipeline,
    steps=steps,
    name="Video Editing v1",
)
print(f"Template saved: {template.name} (id: {template.id})")
print(f"  Version: {template.version}")
print(f"  Steps: {len(template.steps)}")

In [ ]:
# Export to a file for git version control
s.export.pipeline((pipeline, steps), "pipelines/video_editing_v1.json")

In [ ]:
# Clone the template with different inputs
new_pipeline, new_steps = s.pipelines.clone_template(
    template,
    inputs={"video_path": "/data/another_video.mp4", "output_dir": "/data/output2"},
)
s.pipelines.inspect(new_pipeline, new_steps)

## 6. Preview LLM Compiler Prompt

See exactly what context the LLM compiler would receive — useful for
debugging why it generated unexpected steps.

In [ ]:
# Preview the prompt without calling the LLM
prompt = s.compiler.preview_prompt(
    "Resize video to 720p then detect faces on each scene clip",
    inputs={"video_path": "/data/input.mp4"},
)
print(prompt)

In [ ]:
# Clean up
s.close()